In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [ ]:
!pip install -q wandb

import wandb

In [ ]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
wandb.login(key=user_secrets.get_secret("wandb"))

In [1]:
'''import wandb

try:
    # Replace with your actual 40-character API key
    api_key = "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" 
    
    # verify=True forces server-side authentication check
    wandb.login(key=api_key, verify=True)
    print("Success! Your W&B API key is valid.")
    
except Exception as e:
    print(f"Authentication Failed: {e}")
'''

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Success! Your W&B API key is valid.


# **Smart mcq solver**

## milestone-0

In [2]:
%pip install -q transformers datasets accelerate scikit-learn tqdm sentencepiece protobuf gensim ## Install required libraries


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 107.7 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which

In [3]:
# Import all necessary packages

import os
import re
import glob
import warnings

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import pandas as pd
import numpy as np

from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForMultipleChoice,
    get_cosine_schedule_with_warmup
)

warnings.filterwarnings("ignore") # hide warning messages for cleaner output


import sentencepiece #needed for some transformer models (DeBERTa uses SentencePiece tokenization)

os.environ["TOKENIZERS_PARALLELISM"] = "false" #disable tokenizer parallelism warnings

In [4]:
 # Set random seeds for reproducibility

torch.manual_seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Check what computing device is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device name ", device)


device name  cuda


In [5]:
TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"

DEBERTA_MODEL_PATH = "microsoft/deberta-v3-base"
DISTILBERT_MODEL_PATH = "distilbert-base-uncased"

SCRATCH_MODEL_WEIGHTS = "best_model_scratch.pt" ## saved model weights 


# training flags
RUN_TRAINING_FROM_SCRATCH = True
RUN_TRANSFORMER_TRAINING = True

# ANN hyperparameters
ANN_EPOCHS = 5
ANN_BATCH_SIZE = 32
ANN_LR = 0.001
ANN_EMBED_DIM = 256
ANN_HIDDEN_DIM = 256

# Transformer Hyperparameters 
TRANSFORMER_EPOCHS = 10
TRANSFORMER_BATCH_SIZE = 2
DEBERTA_LR = 5e-6
DISTILBERT_LR = 2e-5
GRAD_ACCUM_STEPS = 8

# ensemble threshold 
MIN_USABLE_MAP3 = 0.40

LABEL_MAP = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}



In [6]:
# Load the datasets
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

# Basic dataset information
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("Train columns:", train_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())

# Find the answer column
LABEL_COL = None

for col in ["answer", "Answer", "label", "Label", "correct_answer"]:
    if col in train_df.columns:
        LABEL_COL = col
        break

if LABEL_COL:
    print("Answer column:", LABEL_COL)
    print(train_df[LABEL_COL].value_counts())
else:
    print("No answer column found.")
    RUN_TRAINING_FROM_SCRATCH = False
    RUN_TRANSFORMER_TRAINING = False

Train shape: (2000, 8)
Test shape: (500, 7)
Train columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']
Test columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E']
Answer column: answer
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64


In [7]:
train_df.head(2)

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A


In [8]:
# Remove duplicate questions
train_df_clean = train_df.drop_duplicates(
    subset=["prompt", "A", "B", "C", "D", "E"]
)

print("Original dataset size:", len(train_df))
print("After removing duplicates:", len(train_df_clean))
print("Duplicates removed:", len(train_df) - len(train_df_clean))

Original dataset size: 2000
After removing duplicates: 1817
Duplicates removed: 183


In [9]:
# split the data into training and validation sets
gss = GroupShuffleSplit(test_size=0.1, n_splits=1, random_state=42)

train_idx, val_idx = next(
    gss.split(train_df, groups=train_df["prompt"])
)

t_df = train_df.iloc[train_idx]
v_df = train_df.iloc[val_idx]

print("Training samples:", len(t_df))
print("Validation samples:", len(v_df))


Training samples: 1806
Validation samples: 194


## Milestone 1 — (NLP Foundation & Semantic Similarity)

In [10]:
# Clean text by converting to lowercase and removing special characters
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [11]:
# Check for missing data
print(train_df.isnull().sum())
print(test_df.isnull().sum())

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
dtype: int64


In [12]:
def calculate_map3(logits, targets):
    total_score = 0

    
    for i in range(len(logits)):

        # Get indices of top 3 highest scores
        top3 = np.argsort(logits[i])[-3:][::-1]

        # Check where the correct answer appears
        if targets[i] == top3[0]:
            total_score += 1.0
        elif targets[i] == top3[1]:
            total_score += 0.5
        elif targets[i] == top3[2]:
            total_score += 1/3
        else:
            total_score += 0

    # Return average score
    return total_score / len(logits)

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Build TF-IDF vocabulary
all_text = []

for col in ["prompt", "A", "B", "C", "D", "E"]:
    for text in train_df[col]:
        all_text.append(clean_text(str(text)))

tfidf_vectorizer = TfidfVectorizer(max_features=5000,stop_words="english")

tfidf_vectorizer.fit(all_text)

print("TF-IDF vocabulary size:", len(tfidf_vectorizer.vocabulary_))


# Predict answers using cosine similarity
def tfidf_predict(df, vectorizer):

    predictions = []

    for _, row in df.iterrows():

        # Convert prompt to TF-IDF vector
        prompt_vec = vectorizer.transform([clean_text(row["prompt"])])

        scores = []

        # Compare prompt with each option
        for option in ["A", "B", "C", "D", "E"]:

            option_vec = vectorizer.transform([clean_text(row[option])])

            similarity = cosine_similarity(prompt_vec, option_vec)[0][0]

            scores.append(similarity)

        predictions.append(scores)

    return np.array(predictions)


tfidf_preds = tfidf_predict(v_df, tfidf_vectorizer)

tfidf_targets = v_df[LABEL_COL].map(LABEL_MAP).values

tfidf_map3 = calculate_map3(tfidf_preds, tfidf_targets)

print("TF-IDF map3:", round(tfidf_map3, 4))

TF-IDF vocabulary size: 2761
TF-IDF mAP@3: 0.25


In [14]:
from gensim.models import Word2Vec

# Prepare text for Word2Vec
sentences = []

for col in ["prompt", "A", "B", "C", "D", "E"]:
    for text in train_df[col].astype(str):

        tokens = clean_text(text).split()

        if len(tokens) > 0:
            sentences.append(tokens)

# Train the Word2Vec model
w2v_model = Word2Vec(sentences=sentences,vector_size=100,window=5,min_count=1,workers=4,epochs=10,seed=42)
print("Vocabulary size:", len(w2v_model.wv))


# Convert text into a vector
def text_to_w2v_vector(text, model):

    tokens = clean_text(text).split()

    vectors = []

    for token in tokens:
        if token in model.wv:
            vectors.append(model.wv[token])

    if len(vectors) > 0:
        return np.mean(vectors, axis=0)

    return np.zeros(model.vector_size)


# Predict answers using cosine similarity
def w2v_predict(df, model):

    predictions = []

    for _, row in df.iterrows():

        prompt_vec = text_to_w2v_vector(row["prompt"], model).reshape(1, -1)

        scores = []

        for option in ["A", "B", "C", "D", "E"]:

            option_vec = text_to_w2v_vector(row[option], model).reshape(1, -1)

            similarity = cosine_similarity(prompt_vec, option_vec)[0][0]

            scores.append(similarity)

        predictions.append(scores)

    return np.array(predictions)


w2v_preds = w2v_predict(v_df, w2v_model)

w2v_targets = v_df[LABEL_COL].map(LABEL_MAP).values

w2v_map3 = calculate_map3(w2v_preds, w2v_targets)

print("Word2vec map3:", round(w2v_map3, 4))

Vocabulary size: 2973
Word2Vec mAP@3: 0.3299


## Milestone 2 (Enter the Transformers)

In [15]:
# Load the DeBERTa tokenizer
tokenizer_demo = AutoTokenizer.from_pretrained(DEBERTA_MODEL_PATH)

# Tokenize a sample question
sample_text = train_df["prompt"].iloc[0]

simple_tokens = clean_text(sample_text).split()
deberta_tokens = tokenizer_demo.tokenize(sample_text)

print("Simple tokens:", simple_tokens[:10])
print("DeBERTa tokens:", deberta_tokens[:10])

# Convert tokens to IDs
token_ids = tokenizer_demo.convert_tokens_to_ids(deberta_tokens[:10])
print("Token IDs:", token_ids)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Simple tokens: ['pick', 'the', 'best', 'possible', 'answer', 'what', 'is', 'martin', 'heidegger', 's']
DeBERTa tokens: ['▁Pick', '▁the', '▁best', '▁possible', '▁answer', ':', '▁What', '▁is', '▁Martin', '▁Heidegger']
Token IDs: [8498, 262, 410, 628, 1330, 294, 458, 269, 2963, 49965]


In [16]:
# Load the pre-trained model
model_demo = AutoModel.from_pretrained(DEBERTA_MODEL_PATH).to(device)
model_demo.eval()

def get_transformer_embedding(text, tokenizer, model, max_len=128):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_len
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    return outputs.last_hidden_state.mean(dim=1).cpu().numpy()

sample_row = train_df.iloc[0]

prompt_emb = get_transformer_embedding(
    str(sample_row["prompt"])[:200],
    tokenizer_demo,
    model_demo
)

print("Embedding shape:", prompt_emb.shape)

for col in ["A", "B", "C", "D", "E"]:
    opt_emb = get_transformer_embedding(
        str(sample_row[col])[:200],
        tokenizer_demo,
        model_demo
    )

    sim = cosine_similarity(prompt_emb, opt_emb)[0][0]
    print(f"{col}: {sim:.4f}")

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Embedding shape: (1, 768)
A: 0.3673
B: 0.5305
C: 0.9250
D: 0.3770
E: 0.8692


In [17]:
# Zero-shot prediction using transformer embeddings
def transformer_embedding_predict(df, tokenizer, model, max_len=128):
    predictions = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Zero-shot predictions"):
        prompt_emb = get_transformer_embedding(
            str(row["prompt"])[:300],
            tokenizer,
            model,
            max_len
        )

        option_sims = []
        for col in ["A", "B", "C", "D", "E"]:
            opt_emb = get_transformer_embedding(
                str(row[col])[:300],
                tokenizer,
                model,
                max_len
            )

            sim = cosine_similarity(prompt_emb, opt_emb)[0][0]
            option_sims.append(sim)

        predictions.append(option_sims)

    return np.array(predictions)

zs_preds = transformer_embedding_predict(v_df, tokenizer_demo, model_demo)
zs_targets = v_df[LABEL_COL].map(LABEL_MAP).values

zs_map3 = calculate_map3(zs_preds, zs_targets)

print("Zero-shot mAP@3:", round(zs_map3, 4))

del model_demo

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Zero-shot predictions:   0%|          | 0/194 [00:00<?, ?it/s]

Zero-shot mAP@3: 0.4218


## Milestone 3 (Context Augmentation with RAG Pipelines)

In [18]:
# Build the vector database
rag_vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

train_prompt_vectors = rag_vectorizer.fit_transform(
    t_df["prompt"].astype(str).apply(clean_text)
)

print("Training questions:", train_prompt_vectors.shape[0])
print("Feature size:", train_prompt_vectors.shape[1])

Training questions: 1806
Feature size: 796


In [19]:
# Find the most similar questions from the training data

def retrieve_similar(query, vectorizer, vectors, source_df, top_k=3):
    
    query_vec = vectorizer.transform([clean_text(query)]) # Convert the query into a TF-IDF vector

    # Calculate cosine similarity with all training questions
    similarities = cosine_similarity(query_vec, vectors)[0]     # Calculate cosine similarity with all training questions


    top_indices = similarities.argsort()[-top_k:][::-1]     # Get the indices of the top matching questions

    results = []
    for idx in top_indices:
        results.append({
            "prompt": source_df.iloc[idx]["prompt"],
            "answer": source_df.iloc[idx][LABEL_COL],
            "similarity": similarities[idx]
        })

    return results

sample_query = str(test_df["prompt"].iloc[0])# Retrieve similar questions for one test example

similar_questions = retrieve_similar(
    sample_query,
    rag_vectorizer,
    train_prompt_vectors,
    t_df
)

print("Sample query:")
print(sample_query[:150] + "...")

print("Most similar questions:")
for i, s in enumerate(similar_questions, 1):
    print(f"{i}. Similarity: {s['similarity']:.3f}")
    print("Ans:", s["answer"])
    print(s["prompt"][:120] + "...")

Sample query:
Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quantum mechanics? carefully....
Most similar questions:
1. Similarity: 1.000
Ans: A
Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quant...
2. Similarity: 1.000
Ans: A
Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quant...
3. Similarity: 0.979
Ans: A
Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quant...


In [20]:
# Retrieve similar questions for a sample test question
sample_row = test_df.iloc[0]
sample_prompt = str(sample_row["prompt"])

retrieved = retrieve_similar(
    sample_prompt,
    rag_vectorizer,
    train_prompt_vectors,
    t_df,
    top_k=2
)


augmented_prompt = "Retrieved Context" # create an augmented prompt by adding retrieved context

for i, r in enumerate(retrieved, 1):
    augmented_prompt += (
        f"Similar Question {i} (Answer: {r['answer']}): "
        f"{r['prompt'][:100]}..."
    )

augmented_prompt += f"Current Question{sample_prompt[:200]}"

print(augmented_prompt)

Retrieved ContextSimilar Question 1 (Answer: A): Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in ...Similar Question 2 (Answer: A): Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in ...Current QuestionPick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quantum mechanics? carefully.


In [21]:
# add retrieved context to each question
def augment_dataframe(df, is_train=False):
    augmented_prompts = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Augmenting prompts"):
        query = str(row["prompt"])

        top_k = 2 if is_train else 1
        retrieved = retrieve_similar(
            query,
            rag_vectorizer,
            train_prompt_vectors,
            t_df,
            top_k=top_k
        )

        # Skip the same question when creating the training set
        if is_train and len(retrieved) > 1 and retrieved[0]["prompt"] == query:
            context_q = retrieved[1]
        else:
            context_q = retrieved[0]

        aug_prompt = (
            f"Context: {context_q['prompt'][:100]}... "
            f"Answer: {context_q['answer']}. "
            f"Question: {query}"
        )

        augmented_prompts.append(aug_prompt)

    df_aug = df.copy()
    df_aug["prompt"] = augmented_prompts

    return df_aug

# Update the datasets with augmented prompts
t_df = augment_dataframe(t_df, is_train=True)
v_df = augment_dataframe(v_df)
test_df = augment_dataframe(test_df)

Augmenting prompts:   0%|          | 0/1806 [00:00<?, ?it/s]

Augmenting prompts:   0%|          | 0/194 [00:00<?, ?it/s]

Augmenting prompts:   0%|          | 0/500 [00:00<?, ?it/s]

# Milestone 4 — Formulating MCQ Task & Fine-Tuning

## Simple Vocabulary Class & Converts words into integer IDs

In [22]:
class Vocabulary:
    def __init__(self, max_tokens=10000, max_len=128):
        # Maximum number of words to keep
        self.max_tokens = max_tokens

        # Maximum sequence length
        self.max_len = max_len

        # Special tokens
        self.word2idx = {
            "<PAD>": 0,
            "<UNK>": 1
        }

        self.idx2word = {
            0: "<PAD>",
            1: "<UNK>"
        }

        # Count how many times each word appears
        self.word_counts = Counter()

    # Read all text and build the vocabulary
    def fit(self, texts):

        # Count every word
        for text in texts:
            words = clean_text(text).split()
            self.word_counts.update(words)

        # Keep only the most common words
        common_words = self.word_counts.most_common(self.max_tokens - 2)

        # Give every word a unique ID
        for word, _ in common_words:
            if word not in self.word2idx:
                index = len(self.word2idx)
                self.word2idx[word] = index
                self.idx2word[index] = word

    # Convert a sentence into a list of IDs
    def transform(self, text):

        words = clean_text(text).split()

        ids = []

        # Replace each word with its ID
        for word in words:
            ids.append(self.word2idx.get(word, 1))   # 1 = <UNK>

        # Pad short sentences
        if len(ids) < self.max_len:
            padding = [0] * (self.max_len - len(ids))
            ids += padding

        # Cut long sentences
        else:
            ids = ids[:self.max_len]

        return ids

    # Return vocabulary size
    def __len__(self):
        return len(self.word2idx)


# build Vocabulary from Training Data

all_training_texts = []

# Add question text
all_training_texts.extend(t_df["prompt"])

# Add all five options
all_training_texts.extend(t_df["A"])
all_training_texts.extend(t_df["B"])
all_training_texts.extend(t_df["C"])
all_training_texts.extend(t_df["D"])
all_training_texts.extend(t_df["E"])

# Create vocabulary object
vocab = Vocabulary(
    max_tokens=15000,
    max_len=128
)

# Learn all words from training data
vocab.fit(all_training_texts)

print("Vocabulary size:", len(vocab))

Vocabulary size: 3139


## Dataset Class for the ANN Model & This prepares one question at a time

In [23]:
class MCQDataset(Dataset):

    def __init__(self, df, vocab, is_test=False):
        # Store the dataset
        self.df = df

        # Vocabulary object for converting text to IDs
        self.vocab = vocab

        # True when working with test data
        self.is_test = is_test

    # Total number of samples
    def __len__(self):
        return len(self.df)

    # Get one sample from the dataset
    def __getitem__(self, idx):

        # Read one row
        row = self.df.iloc[idx]

        # Convert the question into token IDs
        prompt = self.vocab.transform(row["prompt"])
        prompt_ids = torch.tensor(prompt, dtype=torch.long)

        # Convert all five options into token IDs
        option_ids = []

        for option in ["A", "B", "C", "D", "E"]:
            ids = self.vocab.transform(row[option])
            option_ids.append(ids)

        option_ids = torch.tensor(option_ids, dtype=torch.long)

        # Store everything in a dictionary
        sample = {
            "prompt": prompt_ids,
            "options": option_ids
        }

        # Training/validation data also contains the correct answer
        if not self.is_test:

            label_col = LABEL_COL if LABEL_COL in row else "answer"

            correct_answer = row[label_col]

            sample["label"] = torch.tensor(
                LABEL_MAP[correct_answer],
                dtype=torch.long
            )

        return sample

## Dataset Class for Transformer Models & Converts each MCQ into transformer inputs

In [24]:
class TransformerMCQDataset(Dataset):

    def __init__(self, df, tokenizer, max_len=128, is_test=False):

        # Store dataset
        self.df = df.reset_index(drop=True)

        # Save tokenizer
        self.tokenizer = tokenizer

        # Maximum sequence length
        self.max_len = max_len

        # Check whether this is test data
        self.is_test = is_test

        # Find the label column (only for training data)
        if not self.is_test:

            self.label_col = None

            possible_labels = [
                "answer",
                "Answer",
                "label",
                "Label",
                "correct_answer"
            ]

            for col in possible_labels:
                if col in self.df.columns:
                    self.label_col = col
                    break

    # Number of samples
    def __len__(self):
        return len(self.df)

    # Return one sample
    def __getitem__(self, idx):

        # Read one row
        row = self.df.iloc[idx]

        # Question text
        prompt = str(row["prompt"])

        # Collect all five answer choices
        choices = []

        for option in ["A", "B", "C", "D", "E"]:
            choices.append(str(row[option]))

        # Tokenize question with each option
        encoded = self.tokenizer(
            [prompt] * 5,
            choices,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        # Store model inputs
        sample = {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0)
        }

        # Some models (like BERT) also return token_type_ids
        if "token_type_ids" in encoded:
            sample["token_type_ids"] = encoded["token_type_ids"].squeeze(0)

        # add label for training/validation data
        if not self.is_test:

            answer = str(row[self.label_col]).strip().upper()

            sample["label"] = torch.tensor(
                LABEL_MAP.get(answer, 0),
                dtype=torch.long
            )

        return sample

## ANN Model for Multiple Choice Questions

In [25]:
class ANNMatchingModel(nn.Module):

    def __init__(self, vocab_size, embed_dim=256, hidden_dim=256, dropout=0.3):
        super().__init__()

        # Convert word IDs into dense vectors
        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=0
        )

        # Small feature extraction network
        self.fc = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        # Normalize features
        self.layer_norm = nn.LayerNorm(hidden_dim)

        # Final prediction network
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 4, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(dropout / 2),

            nn.Linear(128, 1)
        )

    # Convert a sentence into one feature vector
    def _encode(self, ids):

        # Ignore padding tokens
        mask = (ids != 0).float().unsqueeze(-1)

        # Get word embeddings
        embeddings = self.embedding(ids)

        # Remove padding influence
        embeddings = embeddings * mask

        # Add all embeddings together
        total = torch.sum(embeddings, dim=1)

        # Count valid (non-padding) words
        count = torch.clamp(torch.sum(mask, dim=1), min=1.0)

        # Average embedding
        pooled = total / count

        # Extract useful features
        features = self.fc(pooled)

        return self.layer_norm(features)

    def forward(self, prompt_ids, options_ids):

        batch_size = prompt_ids.size(0)

        # Encode the question
        question_features = self._encode(prompt_ids)

        # Encode all answer choices
        option_features = self._encode(
            options_ids.view(batch_size * 5, -1)
        )

        option_features = option_features.view(batch_size, 5, -1)

        # Copy question features for every option
        question_features = question_features.unsqueeze(1)
        question_features = question_features.expand(-1, 5, -1)

        # Combine question and option features
        combined_features = torch.cat([
            question_features,
            option_features,
            torch.abs(question_features - option_features),
            question_features * option_features
        ], dim=-1)

        # Predict one score for each option
        scores = self.classifier(combined_features)

        return scores.squeeze(-1)

## Train the ANN Model

In [26]:
# Create training and validation datasets
train_ds = MCQDataset(t_df, vocab)
val_ds = MCQDataset(v_df, vocab)

# Create dataloaders
train_dl = DataLoader(
    train_ds,
    batch_size=ANN_BATCH_SIZE,
    shuffle=True
)

val_dl = DataLoader(
    val_ds,
    batch_size=ANN_BATCH_SIZE * 2,
    shuffle=False
)

# Create the ANN model
model_scratch = ANNMatchingModel(
    len(vocab),
    ANN_EMBED_DIM,
    ANN_HIDDEN_DIM
).to(device)

# Loss function
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Optimizer
optimizer = optim.AdamW(
    model_scratch.parameters(),
    lr=ANN_LR,
    weight_decay=1e-4
)

# Learning rate scheduler
scheduler = CosineAnnealingLR(
    optimizer,
    T_max=ANN_EPOCHS,
    eta_min=1e-5
)

best_scratch_map3 = 0.0

##wandb
wandb.init(
    project="smart-mcq-solver",
    name="ANN",
    config={
        "epochs": ANN_EPOCHS,
        "batch_size": ANN_BATCH_SIZE,
        "lr": ANN_LR,
        "embed_dim": ANN_EMBED_DIM,
        "hidden_dim": ANN_HIDDEN_DIM,
    }
)

wandb.watch(model_scratch)

# Training Loop
for epoch in range(1, ANN_EPOCHS + 1):

    model_scratch.train()
    total_loss = 0

    for batch in tqdm(train_dl, desc=f"Epoch {epoch}"):

        prompt = batch["prompt"].to(device)
        options = batch["options"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        outputs = model_scratch(prompt, options)

        loss = criterion(outputs, labels)

        loss.backward()

        # Prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(
            model_scratch.parameters(),
            1.0
        )

        optimizer.step()

        total_loss += loss.item() * prompt.size(0)

    scheduler.step()

    # Validation

    model_scratch.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for batch in val_dl:

            prompt = batch["prompt"].to(device)
            options = batch["options"].to(device)
            labels = batch["label"].to(device)

            outputs = model_scratch(prompt, options)

            all_preds.append(outputs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    map3_score = calculate_map3(all_preds, all_labels)

    print(f"Epoch {epoch}: mAP@3 = {map3_score:.4f}")
    wandb.log({
    "epoch": epoch,
    "train_loss": total_loss / len(train_ds),
    "val_map3": map3_score,
    "learning_rate": optimizer.param_groups[0]["lr"]
     })

    # Save the best model
    if map3_score > best_scratch_map3:

        best_scratch_map3 = map3_score

        torch.save(
            model_scratch.state_dict(),
            SCRATCH_MODEL_WEIGHTS
        )

print(f"best validation mAP@3 {best_scratch_map3:.4f}")

wandb.finish()

Epoch 1:   0%|          | 0/57 [00:00<?, ?it/s]

Epoch 1: mAP@3 = 0.8643


Epoch 2:   0%|          | 0/57 [00:00<?, ?it/s]

Epoch 2: mAP@3 = 0.9347


Epoch 3:   0%|          | 0/57 [00:00<?, ?it/s]

Epoch 3: mAP@3 = 0.9648


Epoch 4:   0%|          | 0/57 [00:00<?, ?it/s]

Epoch 4: mAP@3 = 0.9708


Epoch 5:   0%|          | 0/57 [00:00<?, ?it/s]

Epoch 5: mAP@3 = 0.9768
best validation mAP@3 0.9768


## Train function Transformer Model


In [27]:
def train_transformer(
    model_name,
    save_path,
    batch_size=2,
    epochs=10,
    lr=2e-5,
    max_len=128
):

    print(f"\nStarting training: {model_name}")
    print(f"Epochs: {epochs}")
    print(f"Batch Size: {batch_size}")
    print(f"Effective Batch Size: {batch_size * GRAD_ACCUM_STEPS}")
    print(f"Learning Rate: {lr}")

    # Load tokenizer and model

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModelForMultipleChoice.from_pretrained(
        model_name,
        ignore_mismatched_sizes=True
    )

    model = model.float().to(device)
    
    # Create datasets

    train_ds = TransformerMCQDataset(
        t_df,
        tokenizer,
        max_len=max_len
    )

    val_ds = TransformerMCQDataset(
        v_df,
        tokenizer,
        max_len=max_len
    )

    train_dl = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True
    )

    val_dl = DataLoader(
        val_ds,
        batch_size=batch_size * 2,
        shuffle=False
    )

    # Optimizer and Scheduler

    optimizer = optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=0.01
    )

    total_steps = (len(train_dl) // GRAD_ACCUM_STEPS) * epochs
    warmup_steps = max(1, total_steps // 10)

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        warmup_steps,
        total_steps
    )

    scaler = torch.cuda.amp.GradScaler()

    best_map3 = 0.0

    # Training Loop

    for epoch in range(1, epochs + 1):

        model.train()

        total_loss = 0
        valid_batches = 0

        optimizer.zero_grad()

        for step, batch in enumerate(
            tqdm(train_dl, desc=f"Epoch {epoch}/{epochs}")
        ):

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            token_type_ids = batch.get("token_type_ids")

            if token_type_ids is not None:
                token_type_ids = token_type_ids.to(device)

            # Forward pass
            with torch.cuda.amp.autocast():

                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    token_type_ids=token_type_ids,
                    labels=labels
                )

                loss = outputs.loss

            # Skip invalid loss
            if torch.isnan(loss):
                continue

            # Backpropagation
            scaler.scale(loss / GRAD_ACCUM_STEPS).backward()

            total_loss += loss.item() * input_ids.size(0)
            valid_batches += 1

            # Update model after gradient accumulation
            if (
                (step + 1) % GRAD_ACCUM_STEPS == 0
                or
                (step + 1) == len(train_dl)
            ):

                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    1.0
                )

                scaler.step(optimizer)
                scaler.update()

                scheduler.step()

                optimizer.zero_grad()

        # Validation

        model.eval()

        predictions = []
        targets = []

        with torch.no_grad():

            for batch in val_dl:

                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["label"].to(device)

                token_type_ids = batch.get("token_type_ids")

                if token_type_ids is not None:
                    token_type_ids = token_type_ids.to(device)

                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    token_type_ids=token_type_ids,
                    labels=labels
                )

                predictions.append(outputs.logits.cpu().numpy())
                targets.append(labels.cpu().numpy())

        predictions = np.concatenate(predictions)
        targets = np.concatenate(targets)

        map3_score = calculate_map3(
            predictions,
            targets
        )

        print(f"Epoch {epoch}/{epochs} | mAP@3: {map3_score:.4f}")

        # Save the best model
        if map3_score > best_map3:

            best_map3 = map3_score

            model.save_pretrained(save_path + "_folder")
            tokenizer.save_pretrained(save_path + "_folder")

            print("Best model saved!")

    print(f"\nBest Validation mAP@3: {best_map3:.4f}")

    return best_map3



## Train DeBERTa & DistilBERT Model 


In [ ]:

best_deberta_map3 = train_transformer(
    model_name=DEBERTA_MODEL_PATH,
    save_path="deberta_model",
    batch_size=TRANSFORMER_BATCH_SIZE,
    epochs=TRANSFORMER_EPOCHS,
    lr=DEBERTA_LR,
    max_len=128
)


best_distilbert_map3 = train_transformer(
    model_name=DISTILBERT_MODEL_PATH,
    save_path="distilbert_model",
    batch_size=TRANSFORMER_BATCH_SIZE,
    epochs=TRANSFORMER_EPOCHS,
    lr=DISTILBERT_LR,
    max_len=128
)


Starting training: microsoft/deberta-v3-base
Epochs: 10
Batch Size: 2
Effective Batch Size: 16
Learning Rate: 5e-06


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                

Epoch 1/10:   0%|          | 0/903 [00:00<?, ?it/s]

Epoch 1/10 | mAP@3: 0.5430


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best model saved!


Epoch 2/10:   0%|          | 0/903 [00:00<?, ?it/s]